In [ ]:
import pandas as pd
from scipy import stats
from scipy.stats import sem, skew, kurtosis
import seaborn as sns

import matplotlib.pyplot as plt


In [ ]:
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('../data/output.csv')
df.head()

In [ ]:
df['iteration_time'] = df['sample_measured_value'] / df['iteration_count']
df.head()

In [ ]:
UNIT_TO_SECONDS = {
    'ns': 1e-9,
    'us': 1e-6,
    'µs': 1e-6,
    'ms': 1e-3,
    's': 1.0,
}

df['throughput [elem/s]'] = (1 / (df['iteration_time'] * df['unit'].map(UNIT_TO_SECONDS))) * df['throughput_num']
df.head()

In [ ]:
GROUP_KEYS = ['benchmark', 'backend', 'function', 'version', 'suffix']


def benchmark_stats(group, col='throughput [elem/s]'):
    x = group[col]
    return pd.Series({
        # Położenie
        'mean': x.mean(),
        'median': x.median(),
        'mode': x.mode().iloc[0] if not x.mode().empty else float('nan'),
        'trimmed_mean': stats.trim_mean(x, 0.1),

        # Rozrzut
        'std': x.std(),
        'var': x.var(),
        'sem': sem(x),
        'cv': x.std() / x.mean(),
        'iqr': x.quantile(0.75) - x.quantile(0.25),

        # Percentyle
        'p5': x.quantile(0.05),
        'p25': x.quantile(0.25),
        'p75': x.quantile(0.75),
        'p95': x.quantile(0.95),
        'min': x.min(),
        'max': x.max(),

        # Kształt rozkładu
        'skewness': skew(x),
        'kurtosis': kurtosis(x),
    })


df_stats = df.groupby(GROUP_KEYS, dropna=False).apply(benchmark_stats).reset_index()
df_stats.head()

In [ ]:
parts = df_stats['version'].str.extract(r'^(?P<block_bits>\d+)_(?P<key_bits>\d+)$').astype(int)

df_stats = df_stats.join(parts)
df_stats['word_bits'] = df_stats['block_bits'] // 2
df_stats['key_words'] = df_stats['key_bits'] // df_stats['word_bits']

SPECK_ROUNDS = {
    (32, 64): 22,
    (48, 72): 22,
    (48, 96): 23,
    (64, 96): 26,
    (64, 128): 27,
    (96, 96): 28,
    (96, 144): 29,
    (128, 128): 32,
    (128, 192): 33,
    (128, 256): 34,
}

variants = list(zip(df_stats['block_bits'], df_stats['key_bits']))
df_stats['rounds'] = [SPECK_ROUNDS.get(v) for v in variants]

if df_stats['rounds'].isna().any():
    missing = sorted(set(v for v, r in zip(variants, df_stats['rounds']) if pd.isna(r)))
    raise ValueError('Brak definicji liczby rund dla wariantów: {}'.format(missing))

df_stats.head()

In [ ]:
print(df_stats.columns)

In [ ]:
backend_order = ['scalar', 'sse2', 'avx2', 'avx512']
version_order = ['32_64', '48_72', '48_96', '64_96', '64_128', '96_96', '96_144', '128_128', '128_192', '128_256']
benchmark_order = ['speck', 'engine']

In [ ]:
mean = df_stats[['benchmark', 'backend', 'function', 'version', 'suffix', 'mean']].copy()
mean.head()

In [ ]:
engine_backend = mean[(mean['benchmark'] == 'engine')].copy()
engine_backend.head()

In [ ]:
engine_backend_function_mean = engine_backend.groupby(['backend', 'function'])['mean'].mean().reset_index()
scalar_function_mean = engine_backend_function_mean[engine_backend_function_mean['backend'] == 'scalar'].set_index('function')[
    'mean']
engine_backend_function_mean['speedup'] = engine_backend_function_mean.apply(
    lambda r: r['mean'] / scalar_function_mean[r['function']], axis=1)
engine_backend_function_mean

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
sns.barplot(data=engine_backend_function_mean, x='backend', y='mean', hue='function',
            order=backend_order, ax=ax1)
ax1.set_title('Przepustowość')
ax1.set_ylabel('Średnia [klucze/s]')
ax1.set_xlabel('')
handles, labels = ax1.get_legend_handles_labels()
labels = ['deszyfrowanie', 'szyfrowanie', 'szyfrowanie (klucz generowany w locie)']
ax1.legend(handles, labels, title='operacje')


sns.barplot(data=engine_backend_function_mean, x='backend', y='speedup', hue='function',
            order=backend_order, ax=ax2)
ax2.axhline(y=1.0, color='black', linestyle='dotted', linewidth=1)
ax2.set_title('Przyspieszenie względem scalar')
ax2.set_xlabel('Architektura obliczeniowa')
ax2.set_ylabel('Przyspieszenie (krotność)')
ax2.get_legend().remove()

plt.show()

In [ ]:
engine_version = mean[(df_stats['benchmark'] == 'engine')].copy()
engine_version.head()

In [ ]:
engine_version_backend_mean = engine_version.groupby(['backend', 'version'])['mean'].mean().reset_index()
scalar_version_mean = engine_version_backend_mean[engine_version_backend_mean['backend'] == 'scalar'].set_index('version')[
    'mean']
engine_version_backend_mean['speedup'] = engine_version_backend_mean.apply(
    lambda r: r['mean'] / scalar_version_mean[r['version']], axis=1)
engine_version_backend_mean

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
sns.barplot(data=engine_version_backend_mean, x='version', y='mean', hue='backend',
            order=version_order, hue_order=backend_order, ax=ax1)
ax1.set_title('Przepustowość')
ax1.set_ylabel('Średnia [klucze/s]')
ax1.set_xlabel('')
ax1.get_legend().set_title('architektura\nobliczeniowa')

sns.barplot(data=engine_version_backend_mean, x='version', y='speedup', hue='backend',
            order=version_order, hue_order=backend_order, ax=ax2)
ax2.axhline(y=1.0, color='black', linestyle='dotted', linewidth=1)
ax2.set_title('Przyspieszenie względem scalar')
ax2.set_xlabel('Wersja algorytmu speck')
ax2.set_ylabel('Przyspieszenie (krotność)')
ax2.get_legend().remove()

plt.show()

In [ ]:
engine_speck = mean[(mean['function'] == 'encrypt_inflight')].copy()
engine_speck.head()

In [ ]:
engine_speck = mean[mean['function'] == 'encrypt_inflight'].copy()

engine = (
    engine_speck[engine_speck['benchmark'] == 'engine']
    .assign(suffix=lambda d: d['suffix'].astype('Int64').astype(str))
    .groupby(['backend', 'suffix'], as_index=False)['mean']
    .mean()
)

speck = (
    engine_speck[engine_speck['benchmark'] == 'speck']
    .groupby(['backend'], as_index=False)['mean']
    .mean()
    .rename(columns={'mean': 'speck_mean'})
)

cmp = engine.merge(speck, on='backend', how='left')
cmp['speedup_vs_speck'] = cmp['mean'] / cmp['speck_mean']

throughput_df = pd.concat([
    cmp.assign(series='engine s=' + cmp['suffix'])[['backend', 'mean', 'series']],
    speck.rename(columns={'speck_mean': 'mean'}).assign(series='speck')[['backend', 'mean', 'series']]
], ignore_index=True)

speed_df = pd.concat([
    cmp.assign(series='engine s=' + cmp['suffix'])[['backend', 'speedup_vs_speck', 'series']],
    speck.assign(speedup_vs_speck=1.0, series='speck')[['backend', 'speedup_vs_speck', 'series']]
], ignore_index=True)

In [ ]:
series_order = ['speck', 'engine s=1', 'engine s=2', 'engine s=3']
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

sns.barplot(data=throughput_df, x='backend', y='mean', hue='series',
            order=backend_order, hue_order=series_order, ax=ax1)
ax1.set_title('Przepustowość')
ax1.set_ylabel('Średnia [klucze/s]')
ax1.set_xlabel('')

sns.barplot(data=speed_df, x='backend', y='speedup_vs_speck', hue='series',
            order=backend_order, hue_order=series_order, ax=ax2)
ax2.axhline(1.0, color='black', linestyle='dotted', linewidth=1)
ax2.set_title('Przyspieszenie względem speck')
ax2.set_xlabel('Architektura obliczeniowa')
ax2.set_ylabel('Przyspieszenie (krotność)')
ax2.set_ylim(0.6, 1.1)
ax2.get_legend().remove()

plt.show()